In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

## Loading the dataset

In [2]:
df = pd.read_csv(
    "../dataset/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)

In [3]:
print(df.shape)

print()

df.info()

print()

df.head()

print()

df["label"].value_counts()

(5572, 2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB




ham     4825
spam     747
Name: label, dtype: int64

In [4]:
df.head(10)

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


### Encoding the target

In [5]:
df["label"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

df.head()

df["label"].value_counts()

0    4825
1     747
Name: label, dtype: int64

In [6]:
df['label'].value_counts()

0    4825
1     747
Name: label, dtype: int64

###  Split the data

In [7]:
X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Create the CountVectorizer

In [8]:
vectorizer = CountVectorizer()

### Learn the Vocabulary

In [9]:
X_train_vectorized = vectorizer.fit_transform(X_train) # It reads every training SMS and builds a vocabulary.

What does fit() do?

It reads every training SMS and builds a vocabulary.

-------------------
Free lottery

Meeting tomorrow

Project discussion

Free offer  

the vocabulary becomes

| Index | Word       |
| ----: | ---------- |
|     0 | discussion |
|     1 | free       |
|     2 | lottery    |
|     3 | meeting    |
|     4 | offer      |
|     5 | project    |
|     6 | tomorrow   |

Notice:

Every word appears only once.
The vectorizer assigns an index to each word.

------------------------------------------------------------------------------------

What does transform() do?

Now each SMS becomes a vector.

Example:

Free offer

becomes

| discussion | free | lottery | meeting | offer | project | tomorrow |
| ---------- | ---: | ------: | ------: | ----: | ------: | -------: |
| 0          |    1 |       0 |       0 |     1 |       0 |        0 |

Now Naive Bayes can understand it.


### Transform the Test Data

In [10]:
X_test_vectorized = vectorizer.transform(X_test)

We do NOT write:

vectorizer.fit_transform(X_test)

because that would learn the vocabulary from the test data.

In [11]:
print(vectorizer.get_feature_names_out()[:20])

['00' '000' '000pes' '008704050406' '0089' '0121' '01223585236'
 '01223585334' '02' '0207' '02072069400' '02073162414' '02085076972' '021'
 '03' '04' '0430' '05' '050703' '0578']


In [12]:
print(X_train_vectorized[:5])

  (0, 3373)	1
  (0, 7439)	1
  (0, 7629)	1
  (0, 3290)	1
  (0, 1861)	1
  (1, 1646)	1
  (1, 5210)	2
  (1, 1919)	1
  (1, 7107)	1
  (1, 4812)	1
  (1, 3629)	1
  (1, 6927)	1
  (1, 2420)	1
  (1, 3608)	1
  (1, 7133)	1
  (1, 4981)	1
  (1, 3873)	1
  (1, 5558)	1
  (1, 390)	1
  (1, 2421)	1
  (1, 2419)	1
  (1, 4645)	1
  (1, 4770)	1
  (1, 6748)	1
  (1, 3759)	1
  :	:
  (2, 6148)	1
  (2, 6648)	1
  (2, 1104)	1
  (3, 4157)	1
  (3, 5933)	1
  (3, 1592)	1
  (3, 5403)	1
  (3, 7601)	1
  (4, 1646)	1
  (4, 3608)	1
  (4, 4157)	1
  (4, 5933)	1
  (4, 6595)	1
  (4, 1607)	1
  (4, 3648)	1
  (4, 1380)	1
  (4, 3209)	1
  (4, 6250)	1
  (4, 6775)	1
  (4, 6859)	1
  (4, 6621)	1
  (4, 1667)	1
  (4, 4864)	1
  (4, 3420)	1
  (4, 2887)	1


Imagine the real dataset.

Vocabulary:

7700 words

One SMS contains maybe

10 words

That means:

7690 zeros

10 ones

If we stored all those zeros, we'd waste a huge amount of memory.

Instead of storing:

1 0 0 0 0 0 0 1 0 0 0 ...

it stores only the non-zero values.

So

(0,3373)    1

means

Row 0

Column 3373

Value = 1

That's all.

In [13]:
X_train_vectorized

<4457x7668 sparse matrix of type '<class 'numpy.int64'>'
	with 58930 stored elements in Compressed Sparse Row format>

In [14]:
type(X_train_vectorized)

scipy.sparse.csr.csr_matrix

In [15]:
pd.DataFrame(
    X_train_vectorized[:10].toarray(),
    columns=vectorizer.get_feature_names_out()
)

,00,000,000pes,008704050406,0089,0121,01223585236,01223585334,02,0207,...,zed,zeros,zhong,zindgi,zoe,zogtorius,zyada,èn,ú1,〨ud
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
pd.set_option("display.max_columns", None)

pd.DataFrame(
    X_train_vectorized[:5].toarray(),
    columns=vectorizer.get_feature_names_out()
).iloc[:, 6600:7000]

,swtheart,syd,symbol,sympathetic,symptoms,syrup,system,systems,t4get2text,t91,ta,table,tablet,tablets,tackle,tacos,tactful,tactless,tahan,tait,taj,take,takecare,taken,takes,takin,taking,talent,talents,talk,talked,talkin,talking,talks,tall,tallahassee,tallent,tampa,tank,tantrum,tap,tape,tariffs,tarot,tarpon,taste,tat,tata,tats,tau,taught,taunton,taxi,taxt,taylor,tayseer,tb,tbs,tc,tcr,tcs,tddnewsletter,tea,teach,teacher,teachers,teaches,teaching,team,teams,tear,tears,tease,teasing,tech,technical,technologies,tee,teenager,teeth,teju,tel,telephone,teletext,tell,telling,tellmiss,tells,telly,telphone,telugu,temales,temp,temper,temple,ten,tenants,tendencies,tenerife,tension,term,terminated,terms,termsapply,ternal,terrible,terrific,terror,terrorist,terry,tescos,tessy,test,tests,tex,text,text82228,textand,textbook,textbuddy,textcomp,texted,textin,texting,textoperator,textpod,texts,tgxxrz,th,than,thandiyachu,thangam,thank,thanks,thanks2,thanksgiving,thanku,thanx,thasa,that,that2worzels,thatmum,thats,the,the4th,theacusations,theater,theatre,thecd,thedailydraw,their,thekingshead,them,themed,themes,themob,then,thenampet,theoretically,theory,theplace,there,theres,these,thesedays,thesis,thesmszone,thet,thew,they,theyre,thgt,thia,thin,thing,things,think,thinked,thinkin,thinking,thinks,thinl,thirtyeight,thirunelvali,this,thk,thkin,thm,thnk,thnx,tho,those,though,thought,thoughts,thousands,thout,thread,threats,three,thriller,throat,through,throw,throwin,throwing,thrown,thru,ths,tht,thts,thuglyfe,thurs,thursday,thus,thx,tick,ticket,tickets,tie,tight,tightly,tigress,tihs,tiime,til,till,time,times,timin,timing,timings,tiny,tip,tips,tired,tiring,tirunelvali,tirupur,tis,tissco,title,titles,tiwary,tix,tke,tkls,tkts,tlk,tlp,tm,tmorow,tmorrow,tmr,tmrw,tms,tmw,tnc,tncs,to,toa,toaday,tobacco,tobed,tocall,toclaim,today,todays,todo,tog,together,tohar,tok,token,toking,told,toledo,tolerance,toll,tom,tomarrow,tome,tomeandsaid,tomo,tomorro,tomorrow,tomorw,tone,tones,tones2u,tones2you,tonexs,tongued,tonight,tonite,tons,tonsolitusaswell,too,took,tookplace,tool,tooo,toot,tooth,toothpaste,top,topic,toplay,topped,toppoly,torch,torrents,tortilla,torture,tosend,toshiba,toss,tot,total,totally,totes,touch,touched,tough,toughest,tour,towards,town,toxic,tp,track,trackmarque,trade,traditions,traffic,train,trained,training,trainners,trains,tram,tranquility,transaction,transcribing,transfer,transfered,transferred,transfr,transfred,transport,trav,travel,traveling,travelled,travelling,treacle,treadmill,treasure,treat,treated,treatin,tree,trial,tried,trip,triple,trips,trishul,triumphed,trivia,trouble,troubleshooting,trouser,true,true18,truffles,truly,truro,trust,truth,try,tryin,trying,ts,tsandcs,tscs,tscs087147403231winawk,tsunami,tsunamis,tt,ttyl,tue,tues,tuesday,tuition,tul,tulip,tunde,tune,turkeys
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

## Now it's time to train our first Naive Bayes model

In [17]:
model = MultinomialNB()

model.fit(X_train_vectorized, y_train) # This is where learning happens.

#The model learns:
#The probability of each class (ham vs spam).
#The probability of each word given each class.

MultinomialNB()

### let's make predictions

In [18]:
y_pred = model.predict(X_test_vectorized)

### Evaluate the model

In [19]:
print(confusion_matrix(y_test, y_pred))

print()

print(classification_report(y_test, y_pred))

print()

print("Accuracy:", accuracy_score(y_test, y_pred))

[[964   2]
 [ 12 137]]

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.99      0.92      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115


Accuracy: 0.9874439461883409


## Confusion Matrix

               Predicted
             Ham    Spam

Actual Ham   964      2
Actual Spam   12     137

The model performs very well overall with high precision and accuracy. However, I would analyze the 12 false negatives because missing spam messages can impact user experience. Depending on the business requirement, I might optimize the model to improve recall, even if it slightly reduces precision

### Hyperparameter Tuning

In [20]:
param_grid = {
    "alpha": [0.1, 0.5, 1.0, 2.0] #values of laplace smoothing
}

### GridsearchCV

In [21]:
from sklearn.model_selection import GridSearchCV

In [22]:
grid_search = GridSearchCV(
    estimator=MultinomialNB(),
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

In [23]:
grid_search.fit(X_train_vectorized, y_train)

GridSearchCV(cv=5, estimator=MultinomialNB(), n_jobs=-1,
             param_grid={'alpha': [0.1, 0.5, 1.0, 2.0]}, scoring='f1')

In [24]:
print(grid_search.best_params_)
print(grid_search.best_score_)

{'alpha': 1.0}
0.9298497043633273


Alpha = 1.0 produced the best average F1-score during 5-fold cross-validation and also 1.0 is the standard Laplace smoothing value.

0.9298497043633273 is NOT the test accuracy.

This is:

Average Cross-Validation F1 Score on the training data.
Huge difference.

In [25]:
#  Finally building the model
best_model = grid_search.best_estimator_

In [26]:
y_pred_best = best_model.predict(X_test_vectorized)

In [27]:
print(confusion_matrix(y_test, y_pred_best))

print()

print(classification_report(y_test, y_pred_best))

print()

print("Accuracy:", accuracy_score(y_test, y_pred_best))

[[964   2]
 [ 12 137]]

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.99      0.92      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115


Accuracy: 0.9874439461883409


GridSearchCV --> Accuracy must increase

That's wrong.

Sometimes the default hyperparameters are already close to optimal.

GridSearchCV then serves as validation, not necessarily improvement.

| Metric    | Baseline |  Tuned |
| --------- | -------: | -----: |
| Accuracy  |   98.74% | 98.74% |
| Precision |     0.99 |   0.99 |
| Recall    |     0.92 |   0.92 |
| F1 Score  |     0.95 |   0.95 |


Your GridSearchCV didn't improve the accuracy. Why would you still keep it?
---> GridSearchCV is not only used to improve performance but also to validate that the chosen hyperparameters generalize well across multiple cross-validation folds. In this case, it confirmed that the default alpha=1.0 was already the best choice among the tested values.

In [28]:
sample_messages = [
    "Congratulations! You have won a free iPhone. Claim now!",
    "Hey, are we meeting at 6 PM today?",
    "URGENT! Your account has won ₹10,000.",
    "Please send me the project report by tomorrow."
]

sample_vectors = vectorizer.transform(sample_messages)

predictions = best_model.predict(sample_vectors)

for message, prediction in zip(sample_messages, predictions):
    label = "Spam" if prediction == 1 else "Ham"
    print(f"Message: {message}")
    print(f"Prediction: {label}")
    print("-" * 50)

Message: Congratulations! You have won a free iPhone. Claim now!
Prediction: Spam
--------------------------------------------------
Message: Hey, are we meeting at 6 PM today?
Prediction: Ham
--------------------------------------------------
Message: URGENT! Your account has won ₹10,000.
Prediction: Spam
--------------------------------------------------
Message: Please send me the project report by tomorrow.
Prediction: Ham
--------------------------------------------------


# Conclusion

- Built a Multinomial Naive Bayes classifier to detect SMS spam.
- Used CountVectorizer to convert text messages into numerical feature vectors.
- Applied an 80-20 train-test split while preventing data leakage by fitting the vectorizer only on the training data.
- Achieved an accuracy of **98.74%** on the test set.
- Evaluated the model using Precision, Recall, F1-score, and the Confusion Matrix due to class imbalance.
- Performed hyperparameter tuning using GridSearchCV.
- The best smoothing parameter was:
  - **alpha = 1.0**
- Hyperparameter tuning confirmed that the default value of `alpha` already provided the best cross-validated performance.
- This project demonstrates how Naive Bayes efficiently solves text classification problems such as spam detection.

# Key Learnings

- Learned how CountVectorizer converts text into numerical features.
- Understood why fit() is applied only on training data.
- Learned how Multinomial Naive Bayes estimates class probabilities.
- Understood the zero-frequency problem and Laplace smoothing.
- Learned why F1-score is preferred over accuracy for imbalanced datasets.
- Understood how GridSearchCV validates hyperparameters.